In [2]:
import os, json
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import RobertaTokenizerFast, RobertaForMaskedLM
from datasets import load_dataset


# -------------------------
# 0) 설정
# -------------------------
@dataclass
class Task2Config:
    runs_root: Path
    model_name: str = "seyonec/ChemBERTa-zinc-base-v1"
    which: str = "attention_output"
    topk: int = 32
    batch_size: int = 32                 # downstream encoding 배치
    max_length: int = 128                # SMILES 길이 제한
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # 노드 빈도 필터 (inverted index)
    min_df_frac: float = 0.001           # 너무 드문 노드 제거 (예: 0.1%)
    max_df_frac: float = 0.20            # 너무 자주 켜지는 노드 제거 (예: 20%)

    out_dirname: str = "task2_outputs"


# -------------------------
# 1) TopKSAE 구현 (checkpoint 호환)
# -------------------------
class TopK(nn.Module):
    def __init__(self, k: int):
        super().__init__()
        self.k = int(k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.k <= 0 or self.k >= x.shape[-1]:
            return x
        vals, idx = torch.topk(x, self.k, dim=-1)
        out = torch.zeros_like(x)
        out.scatter_(-1, idx, vals)
        return out


class TopKSAE(nn.Module):
    def __init__(self, n_inputs: int, n_latents: int, k: int, normalize_inputs: bool = True):
        super().__init__()
        self.n_inputs = int(n_inputs)
        self.n_latents = int(n_latents)
        self.k = int(k)
        self.normalize_inputs = bool(normalize_inputs)

        self.W_enc = nn.Parameter(torch.empty(n_latents, n_inputs))
        self.b_enc = nn.Parameter(torch.zeros(n_latents))
        self.W_dec = nn.Parameter(torch.empty(n_inputs, n_latents))
        self.b_dec = nn.Parameter(torch.zeros(n_inputs))

        nn.init.kaiming_uniform_(self.W_enc, a=np.sqrt(5))
        nn.init.kaiming_uniform_(self.W_dec, a=np.sqrt(5))
        self.act = TopK(k)

    def _normalize_x(self, x: torch.Tensor) -> torch.Tensor:
        if not self.normalize_inputs:
            return x
        d = x.shape[-1]
        eps = 1e-6
        norm = x.norm(dim=-1, keepdim=True).clamp(min=eps)
        return x * (np.sqrt(d) / norm)

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self._normalize_x(x)
        pre = F.linear(x, self.W_enc, self.b_enc)     # [B, n_latents]
        h = self.act(pre)                             # topK sparse
        return h, pre


def load_sae_from_latest(latest_pt: Path, device: str) -> Tuple[TopKSAE, dict]:
    state = torch.load(latest_pt, map_location=device, weights_only=False)
    cfg = state.get("cfg", {})
    n_inputs = int(cfg.get("d_model", cfg.get("n_inputs", 768)))
    n_latents = int(cfg.get("n_latents", 4096))
    k = int(cfg.get("topk", 32))

    ae = TopKSAE(n_inputs=n_inputs, n_latents=n_latents, k=k, normalize_inputs=True).to(device)
    ae.load_state_dict(state["model"])
    ae.eval()
    return ae, state


# -------------------------
# 2) SMILES dataset 로딩 (HF zpn/*)
# -------------------------
class SmilesDataset(Dataset):
    def __init__(self, hf_dataset, tokenizer, smiles_col="smiles", max_length=128):
        self.ds = hf_dataset
        self.tok = tokenizer
        self.smiles_col = smiles_col
        self.max_length = max_length

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx: int):
        smi = self.ds[idx][self.smiles_col]
        enc = self.tok(
            smi,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["smiles"] = smi
        return item


def get_hf_task_dataset(task_name: str):
    if task_name == "bbbp":
        return load_dataset("zpn/bbbp")
    if task_name == "bace":
        return load_dataset("zpn/bace_classification")
    if task_name == "clintox":
        return load_dataset("zpn/clintox")
    raise ValueError("task_name must be one of: bbbp, bace, clintox")


# -------------------------
# 3) 레이어별 pooled latent -> topK 추출
# -------------------------
@torch.no_grad()
def pooled_latent_and_topk(
    model: RobertaForMaskedLM,
    ae: TopKSAE,
    dataloader: DataLoader,
    layer: int,
    which: str,
    topk: int,
    device: str,
) -> Tuple[np.ndarray, np.ndarray, List[str]]:
    """
    returns:
      topk_idx: [N, topk] int64
      topk_val: [N, topk] float32
      smiles:  list length N
    """
    model.eval()
    ae.eval()

    layer_mod = model.roberta.encoder.layer[layer]
    if which == "attention_output":
        target = layer_mod.attention.output
    elif which == "mlp_output":
        target = layer_mod.output
    elif which == "residual_stream":
        target = layer_mod
    else:
        raise ValueError("which must be attention_output / mlp_output / residual_stream")

    captured = {"value": None}

    def hook_fn(module, inp, out):
        captured["value"] = out

    handle = target.register_forward_hook(hook_fn)

    all_idx, all_val, all_smiles = [], [], []

    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        smiles = batch["smiles"]

        _ = model.roberta(input_ids=input_ids, attention_mask=attention_mask)
        acts = captured["value"]
        if isinstance(acts, (tuple, list)):
            acts = acts[0]  # 안전장치

        # SAE token-level encode
        flat = acts.reshape(-1, acts.shape[-1])  # [B*T, d]
        h, _ = ae.encode(flat)                   # [B*T, n_latents]
        h = h.reshape(acts.shape[0], acts.shape[1], -1)  # [B, T, n_latents]

        # mask mean pooling -> [B, n_latents]
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        # pooled topK
        vals, idx = torch.topk(pooled, k=topk, dim=-1)

        all_idx.append(idx.cpu().numpy().astype(np.int64))
        all_val.append(vals.cpu().numpy().astype(np.float32))
        all_smiles.extend(smiles)

    handle.remove()

    topk_idx = np.concatenate(all_idx, axis=0)
    topk_val = np.concatenate(all_val, axis=0)
    return topk_idx, topk_val, all_smiles


def pairwise_overlap(a_idx: np.ndarray, b_idx: np.ndarray) -> List[List[int]]:
    """
    a_idx, b_idx: [N, K]
    returns list of length N with overlap indices (node ids) as python lists
    """
    overlaps = []
    for i in range(a_idx.shape[0]):
        s = set(a_idx[i].tolist())
        t = set(b_idx[i].tolist())
        overlaps.append(sorted(list(s.intersection(t))))
    return overlaps


def build_inverted_index(topk_idx: np.ndarray, n_latents: int) -> Dict[int, List[int]]:
    """
    node_id -> list of sample indices where node appears in topK
    """
    inv = {i: [] for i in range(n_latents)}
    N, K = topk_idx.shape
    for n in range(N):
        for j in topk_idx[n]:
            inv[int(j)].append(n)
    return inv


def filter_nodes_by_df(inv: Dict[int, List[int]], N: int, min_frac: float, max_frac: float) -> List[int]:
    kept = []
    for node, samples in inv.items():
        df = len(samples) / max(1, N)
        if df >= min_frac and df <= max_frac:
            kept.append(node)
    return kept


# -------------------------
# 4) 메인 실행: 여러 레이어 SAE를 동시에 돌리고 overlap 저장
# -------------------------
def run_task2(cfg: Task2Config, task_name: str, split: str = "test", layers: Optional[List[int]] = None):
    device = cfg.device
    runs_root = Path(cfg.runs_root)

    # 체크포인트 자동 탐색
    ckpt_root = runs_root / "checkpoints"
    if not ckpt_root.exists():
        raise FileNotFoundError(f"checkpoints dir not found: {ckpt_root}")

    # layers 지정 없으면: checkpoints 폴더에서 자동 추론
    if layers is None:
        cand = sorted([p for p in ckpt_root.glob(f"layer_*_{cfg.which}/latest.pt")])
        if not cand:
            raise FileNotFoundError(f"No latest.pt found under {ckpt_root}/layer_*_{cfg.which}/latest.pt")
        layers = sorted(list({int(p.parent.name.split("_")[1]) for p in cand}))

    # 모델/토크나이저
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=False)
    model = AutoModelForMaskedLM.from_pretrained(cfg.model_name).to(device)

    model.eval()

    # 데이터
    hf = get_hf_task_dataset(task_name)
    ds = SmilesDataset(hf[split], tokenizer, smiles_col="smiles", max_length=cfg.max_length)
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2)

    out_dir = runs_root / cfg.out_dirname / f"{task_name}_{split}"
    out_dir.mkdir(parents=True, exist_ok=True)

    per_layer = {}
    n_latents_ref = None

    # 레이어별 topK 산출
    for L in layers:
        latest_pt = ckpt_root / f"layer_{L}_{cfg.which}" / "latest.pt"
        if not latest_pt.exists():
            raise FileNotFoundError(f"Missing: {latest_pt}")

        ae, state = load_sae_from_latest(latest_pt, device=device)
        n_latents = int(state.get("cfg", {}).get("n_latents", 4096))
        if n_latents_ref is None:
            n_latents_ref = n_latents
        elif n_latents_ref != n_latents:
            raise ValueError(f"n_latents mismatch: {n_latents_ref} vs {n_latents} (layer {L})")

        topk_idx, topk_val, smiles_list = pooled_latent_and_topk(
            model=model,
            ae=ae,
            dataloader=dl,
            layer=L,
            which=cfg.which,
            topk=cfg.topk,
            device=device,
        )

        per_layer[L] = {
            "topk_idx": topk_idx,
            "topk_val": topk_val,
        }

        # 레이어 단독 inverted index 저장
        inv = build_inverted_index(topk_idx, n_latents=n_latents)
        kept_nodes = filter_nodes_by_df(inv, N=topk_idx.shape[0], min_frac=cfg.min_df_frac, max_frac=cfg.max_df_frac)

        with open(out_dir / f"inverted_index_layer{L}.json", "w", encoding="utf-8") as f:
            json.dump(
                {
                    "layer": L,
                    "which": cfg.which,
                    "N": int(topk_idx.shape[0]),
                    "topk": int(cfg.topk),
                    "min_df_frac": cfg.min_df_frac,
                    "max_df_frac": cfg.max_df_frac,
                    "kept_nodes": kept_nodes,
                    "index": {str(k): inv[k] for k in kept_nodes},  
                },
                f,
                ensure_ascii=False,
            )

        # 레이어별 sample-topk 저장
        np.savez_compressed(
            out_dir / f"sample_topk_layer{L}.npz",
            topk_idx=topk_idx,
            topk_val=topk_val,
        )

    # 레이어 간 overlap (pairwise) 저장
    layers_sorted = sorted(layers)
    N = next(iter(per_layer.values()))["topk_idx"].shape[0]

    overlap_dict = {}
    for i in range(len(layers_sorted)):
        for j in range(i + 1, len(layers_sorted)):
            a = layers_sorted[i]
            b = layers_sorted[j]
            overlaps = pairwise_overlap(per_layer[a]["topk_idx"], per_layer[b]["topk_idx"])
            overlap_dict[f"{a}__{b}"] = overlaps

    # JSONL: sample 단위로 묶어서 저장(붙여넣기/후처리 용이)
    jsonl_path = out_dir / f"task2_samplewise_layers_{'-'.join(map(str,layers_sorted))}.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for n in range(N):
            row = {
                "sample_id": n,
                "smiles": smiles_list[n],
                "which": cfg.which,
                "topk": cfg.topk,
                "layers": {},
                "overlaps": {},
            }
            for L in layers_sorted:
                row["layers"][str(L)] = {
                    "topk_idx": per_layer[L]["topk_idx"][n].tolist(),
                    "topk_val": per_layer[L]["topk_val"][n].tolist(),
                }
            for k, v in overlap_dict.items():
                row["overlaps"][k] = v[n]
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("Saved outputs to:", out_dir)
    print("Samplewise JSONL:", jsonl_path)


# -------------------------
# 5) 실행 예시
# -------------------------
cfg = Task2Config(
    runs_root=Path("/home/tjdgns1502/runs/sae"),
    which="attention_output",
    topk=32,
    batch_size=32,
    max_length=128,
    min_df_frac=0.001,
    max_df_frac=0.20,
)

run_task2(cfg, task_name="bbbp", split="test", layers=[4,5])
# run_task2(cfg, task_name="bace", split="test", layers=[4,5])
# run_task2(cfg, task_name="clintox", split="test", layers=[4,5])


Loading weights:   0%|          | 0/108 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie roberta.embeddings.word_embeddings.weight to lm_head.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie lm_head.bias to lm_head.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
RobertaForMaskedLM LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Saved outputs to: /home/tjdgns1502/runs/sae/task2_outputs/bbbp_test
Samplewise JSONL: /home/tjdgns1502/runs/sae/task2_outputs/bbbp_test/task2_samplewise_layers_4-5.jsonl


In [2]:
import numpy as np
from collections import Counter, defaultdict
import math

npz_path = "/home/tjdgns1502/runs/sae/task2_outputs/bbbp_test/sample_topk_layer5.npz"
topk = 32

data = np.load(npz_path)
topk_idx = data["topk_idx"]  # [N, K]
N, K = topk_idx.shape
assert K == topk

# 1) 노드 등장 빈도(df): node가 topK에 포함된 샘플 수
df = Counter()
for n in range(N):
    for node in topk_idx[n]:
        df[int(node)] += 1

# 2) 노드쌍 co-activation count: 같은 샘플 topK에 함께 등장한 횟수
pair_count = Counter()
for n in range(N):
    nodes = sorted(set(map(int, topk_idx[n].tolist())))
    for i in range(len(nodes)):
        for j in range(i+1, len(nodes)):
            pair_count[(nodes[i], nodes[j])] += 1

# 3) 정규화 지표 계산 함수들
def jaccard(i, j, cij):
    # |A∩B| / |A∪B|
    return cij / (df[i] + df[j] - cij)

def p_j_given_i(i, j, cij):
    return cij / df[i]

def pmi(i, j, cij, eps=1e-12):
    # PMI = log( P(i,j) / (P(i)P(j)) )
    pij = cij / N
    pi = df[i] / N
    pj = df[j] / N
    return math.log((pij + eps) / (pi * pj + eps))

# 4) 너무 희귀/너무 흔한 노드는 해석에 방해가 되니 필터
min_df = max(2, int(0.01 * N))   # 예: 최소 1% 샘플에서 등장
max_df = int(0.30 * N)           # 예: 30% 넘게 등장하면 상수성 의심
valid_nodes = {i for i,c in df.items() if (c >= min_df and c <= max_df)}

# 5) 후보 노드쌍 스코어링
rows = []
for (i, j), cij in pair_count.items():
    if i not in valid_nodes or j not in valid_nodes:
        continue
    if cij < 3:  # 너무 적게 같이 켜지면 우연일 확률 큼
        continue
    rows.append((
        i, j, cij,
        df[i], df[j],
        jaccard(i, j, cij),
        p_j_given_i(i, j, cij),
        p_j_given_i(j, i, cij),
        pmi(i, j, cij),
    ))


rows_by_jacc = sorted(rows, key=lambda x: x[5], reverse=True)
rows_by_pmi  = sorted(rows, key=lambda x: x[8], reverse=True)
rows_by_c    = sorted(rows, key=lambda x: x[2], reverse=True)

def show(title, rows, n=20):
    print("\n==", title, "==")
    print("i  j  co  df_i df_j  jacc   P(j|i) P(i|j)  PMI")
    for r in rows[:n]:
        i,j,c,di,dj,jac,pji,pij,pm = r
        print(f"{i:4d} {j:4d} {c:3d} {di:4d} {dj:4d}  {jac:5.3f}  {pji:6.3f} {pij:6.3f}  {pm:6.3f}")

show("Top pairs by Jaccard (same concept candidates)", rows_by_jacc, 25)
show("Top pairs by PMI (non-trivial association)", rows_by_pmi, 25)
show("Top pairs by co-count (raw co-activation)", rows_by_c, 25)

# 6) “한 샘플에서 중복이 얼마나 심한지” 
# (여기서는 Jaccard가 높은 상위 pair들을 '중복 pair'로 가정)
top_pairs = set((i,j) for i,j,_,_,_,jac,_,_,_ in rows_by_jacc[:200])  # 상위 200개만 사용(조절)
dup_counts = []
for n in range(N):
    nodes = sorted(set(map(int, topk_idx[n].tolist())))
    s = set(nodes)
    c = 0
    for (i,j) in top_pairs:
        if i in s and j in s:
            c += 1
    dup_counts.append(c)

dup_counts = np.array(dup_counts)
print("\n== Sample-level redundancy proxy ==")
print("N:", N)
print("mean #high-association pairs per sample:", float(dup_counts.mean()))
print("median:", float(np.median(dup_counts)))
print("max:", int(dup_counts.max()))


== Top pairs by Jaccard (same concept candidates) ==
i  j  co  df_i df_j  jacc   P(j|i) P(i|j)  PMI
1905 3511   3    3    3  1.000   1.000  1.000   4.220
1181 3376  45   53   45  0.849   0.849  1.000   1.348
1397 2674   5    5    6  0.833   1.000  0.833   3.526
2456 2674   5    5    6  0.833   1.000  0.833   3.526
1644 4045   8    9    9  0.800   0.889  0.889   3.003
 506  707   4    5    4  0.800   0.800  1.000   3.709
1937 2656   4    5    4  0.800   0.800  1.000   3.709
2456 2656   4    5    4  0.800   0.800  1.000   3.709
3340 3376  39   43   45  0.796   0.907  0.867   1.414
1659 2848   6    8    6  0.750   0.750  1.000   3.239
 549 2386  15   15   20  0.750   1.000  0.750   2.322
1122 1914   3    4    3  0.750   0.750  1.000   3.932
1181 3340  40   53   43  0.714   0.755  0.930   1.276
1644 3351   9    9   13  0.692   1.000  0.692   2.753
3351 4045   9   13    9  0.692   0.692  1.000   2.753
3316 3376  33   36   45  0.688   0.917  0.733   1.424
 443 3351  13   19   13  0.684   0.